In [ ]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWFjaGEvRHJvcGJveC8wNF90ZWFjaGluZy9BUjEwNDk4X0VNYUMxL1Byb2dQeVF1YXJ0by9Ob3RlYm9va3M='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

In [ ]:
int(4.436e2), round(4.436e2), float(-68), type(42), type("Hello")

In [ ]:
abs(5), abs(-5), abs(6.01 + 0.01), abs(-6.01 - 0.01)

In [ ]:
max(-3, 5)

In [ ]:
min(-3, 5, 0, -10)

In [ ]:
from math import sqrt
a = sqrt(2)
print(a)

In [ ]:
print(42)     # this prints the value to the screen
b = print(42) # prints the value, and we try to capture the output in the variable `b`
print(b)      # however the function has returned nothing, indicated by the value `None` in Python

In [ ]:
help(abs)

In [ ]:
import math

print(math.exp(1))

In [ ]:
from math import sqrt, sin, pi

print(sqrt(36))
print(sin(pi/2))

In [ ]:
x = 2
print(
    math.log(math.exp(x)), 
    math.exp(math.log(x))
)

In [ ]:
import math

print(dir(math))

In [ ]:
def sq(x):
    return x**2

In [ ]:
print(sq(2))

In [ ]:
u = 3.5
v = sq(u+0.5) - 1.0
print(v)

In [ ]:
# YOUR CODE HERE



print(f(0)) 

# perform other tests:
# YOUR CODE HERE


In [ ]:
def do_things(a_number):
    print(f"Your number is {a_number:.2f} to 2DP")
    print("Squared it is:", a_number**2)
    asqrt = a_number**0.5
    print(f"The square root of {a_number} has been returned")
    return asqrt
    print("This does nothing because return immediately ends the function!")

print("This is not inside the function!")

In [ ]:
x = 5.499025
y = do_things(x)
print("This is after the function finishes!")

In [ ]:
print(y)

In [ ]:
#this "global" variable is seen by the main program
number = 101

#define a new function, the variable named `number` here is not related to the one above
def days_in_n_weeks(number):
    # the temporary "local" variable names in the definition do not see or affect the ones in the main program!
    answer = number*7
    return answer

weeks = 6
# run (or "call") the function:
days = days_in_n_weeks(weeks)
print(days)

In [ ]:
print(number)

In [ ]:
print(answer)

In [ ]:
# YOUR CODE HERE

    
ans = trunc5dp(4321.1234567)
print(ans)

In [ ]:
print(trunc5dp(-4321.1234567))  # truncates (round towards zero)

In [ ]:
def myfunc(temporary_variable_name):
    answer = temporary_variable_name * 2
    return answer


x = 10

# x will be renamed temporary_variable_name inside the function (and only there)
y = myfunc(x)

print(y)

In [ ]:
print(temporary_variable_name)  # this will lead to an error

In [ ]:
print(answer) #so will this

In [ ]:
a = 1

def root(a):
    a = a**0.5 #this a is only a tempory name
    return a

b = 2

print(root(b))
print(a)

In [ ]:
def funky(ar):
    # note the variable bee is not defined!
    ans = ar+bee 
    return ans

print("Here!")

In [ ]:
funky(20)

In [ ]:
def funky(ar, bee):
    ans = ar+bee 
    return ans

b = 8222

funky(778, b)

In [ ]:
from math import pi

# YOUR CODE HERE


print(circle_area(2.0))

In [ ]:
def the_answer():
    return 42


ans = the_answer()
print(ans)

In [ ]:
def quadratic_value(a, b, c, x):
    "Evaluate a quadratic polynomial a*x**2 + b*x + c"
    return a*x**2 + b*x + c


help(quadratic_value)

In [ ]:
print(quadratic_value(2.0, 3.0, 4.0, 0.5))

In [ ]:
import math


def ellipse(a, b):
    ab = a*b
    print(ab)
    return math.pi*ab

In [ ]:
ab = 4  # unrelated to local variable ab in ellipse
print(ab)

In [ ]:
a = 2.0
b = 3.0

A = ellipse(a, b)

In [ ]:
print(A)

#the value of ab defined outside the function remains unchanged!
print(ab)

In [ ]:
import math


def quadratic(a, b, c):
    "return the roots of the quadratic equation a*x**2 + b*x + c = 0."
    discriminant = b**2 - 4*a*c  # assumed to be positive
    offset = math.sqrt(discriminant)
    x1 = (-b + offset)/(2*a)
    x2 = (-b - offset)/(2*a)
    return x1, x2


print(quadratic(4.0, 0.0, -16.0))

In [ ]:
root1, root2 = quadratic(4.0, 0.0, -16.0)
print(root1)
print(root2)

In [ ]:
# YOUR CODE HERE


print(square_geometry(3))

In [ ]:
# Run this block to test your function on a hidden test case!

import check03

check03.test0301(square_geometry)

In [ ]:
# YOUR CODE HERE


#set W and H here

perimeter, area = rectangle_geometry(width, height)
print(perimeter)
print(area)

In [ ]:
# Run this block to test your function on a hidden test case!

import check03
check03.test0302(rectangle_geometry)

In [ ]:
def hello():
    print("Hello World!")


hello()

In [ ]:
greeting = hello()  # the action of printing will still be performed

print(greeting)

In [ ]:
def raisepower(base, exponent=2):
    return base**exponent


print(raisepower(10))
print(raisepower(10, 5))
print(raisepower(2, exponent=10))

In [ ]:
from math import cos, sin, tan


def f_zero_ten(InputFunction):
    print(InputFunction(0.0), InputFunction(10.0))


f_zero_ten(cos)
f_zero_ten(sin)
f_zero_ten(tan)

In [ ]:
from math import cos

def f_zero(fname):
    return fname(0.0)

x=0

f_zero(cos(x))

In [ ]:
import math

answer = "The answer is:" #the variable answer holds a string

answer(42) #trying to pass an argument to the variable attempts to use the above string as a function

In [ ]:
# define your twice() function below:
# YOUR CODE HERE



#function to give to your twice() function
def two_times_x(x):
    return 2*x


# give the function two_times_x as an argument to the twice() function along with the value 10

#a2 = twice(???, ???)
# YOUR CODE HERE


print(a2) #this should be the value of two_times_x(two_times_x(10))

In [ ]:
# Run this block to test your function...

import check03

# Test 1. test function to give to your twice() function
def x_squared(x):
    return x**2

# Test 2. 
from math import sqrt
# the sqrt function will also be used to evaluate sqrt(sqrt(16))

check03.test0303(twice)